In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from typing import List
from openai import OpenAI

from youtube_tool import load_YouTube_df
from tavily_tool import webscrape

In [ ]:
df = pd.read_pickle("pickle")

In [2]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [1]:
load_dotenv()


NameError: name 'load_dotenv' is not defined

In [67]:
def gpt_sentiment_analysis(text, model="gpt-4o"):
    # meshedup_comments = column.str.cat(sep='\n')
    # print(meshedup_comments)
    # print(len(column))
    prompt = f"""Analyze the sentiment of comment and return ONLY a single numerical score for each line:
    - Return 1 for positive sentiment
    - Return 0.5 for neutral sentiment
    - Return -1 for negative sentiment
    

    Comments:
    {text}
    """
    # prompt = f"""Analyze the sentiment of each comment below (separated by break line) and return ONLY a single numerical score for each line:
    # - Return 1 for positive sentiment
    # - Return 0.5 for neutral sentiment
    # - Return -1 for negative sentiment
    
    # Provide ONLY a comma-separated list of numbers, like this: 1, 0.5, -1,1,1,1,1
    # There should be a total of {len(column)} numbers, pls check this at the end
    # Comments:
    # {meshedup_comments}
    # """
    # # prompt = f"""Analyze and clasiffy the sentiment of each comment below (separated by ****) and return ONLY negative, positive, or neutral:
    # Feedback: {meshedup_comments}"""
    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}]
        )
        raw_output = completion.choices[0].message.content
        print(raw_output)
        return raw_output

    except Exception as e:
        print("❌ Error:", e)
        return []
    
# def gpt_sentiment_analysis(comments: pd.Series, 
#                             batch_size: int = 20,  # Adjust based on comment lengths
#                             model: str = "gpt-4o") -> List[float]:

#     all_scores = []
    
#     # Split into batches
#     batches = [comments[i:i + batch_size] 
#               for i in range(0, len(comments), batch_size)]
    
#     for batch_num, batch in enumerate(batches):
#         try:
#             # Concatenate with *** separator
#             combined = batch.str.cat(sep='***')
#             print(combined)
            
#             prompt = f"""ANALYSIS RULES:
#             1. Read comments separated by ####
#             2. For each comment, output ONLY 1 (positive), 0.5 (neutral), or -1 (negative)
#             3. Return EXACTLY {len(batch)} numbers
#             4. Use ONLY this format: num1,num2,...,numN
            
#             Comments:
#             {combined}"""
            
#             response = client.chat.completions.create(
#                 model=model,
#                 messages=[{"role": "user", "content": prompt}],
#                 temperature=0.0
#             )
            
#             # Parse response
#             result = response.choices[0].message.content
#             print(f"Batch {batch_num}: result: {result}")
#             batch_scores = result.split(',')
            
#             # Validate count matches
#             if len(batch_scores) != len(batch):
#                 raise ValueError(f"Batch {batch_num}: Got {len(batch_scores)} scores for {len(batch)} comments")
            
#             all_scores.extend(batch_scores)
            
#         except Exception as e:
#             print(f"Error processing batch {batch_num}: {e}")
#             # Optionally add placeholder values or re-raise
#             all_scores.extend([0.5] * len(batch))  # Neutral as fallback
    
#     return all_scores

In [78]:
def gpt_feedback_classifier(text, model="gpt-4o"):
    prompt = f"""Classify the comment below as either "related" or "not related" based on whether it contains product feedback or suggestions.
    - Return ONLY the exact word "related" if it includes feedback/suggestions about a product.
    - Return ONLY the exact word "not related" otherwise.

    Comment:
    {text}

    """
    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0 
        )
        raw_output = completion.choices[0].message.content
        # print(raw_output)
        return raw_output.strip().lower()

    except Exception as e:
        print("❌ Error:", e)
        return []
    
    

In [5]:
product = "Secretlab Titan Evo 2022 Gaming Chair"

youtube_df = load_YouTube_df(product)
tavily_df = webscrape(product)
df = pd.concat([youtube_df, tavily_df], ignore_index=True)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [7]:
df = pd.read_pickle("pickle/Secretlab_Titan_Evo_2022_Gaming_Chair_youtube.pkl")

In [58]:
test = df["content"].str.cat(sep="\n")
test

'are you curious how much assembly is involved in the brand new secret lab titan evo 2022 stick aroun\nd we\'re about to put the chair together hey guys it\'s ryan with btod.com due to popular demand we ha\nve ordered a brand new secret lab titan evo 2022 we do have plans to do reviews and comparisons on t\nhis product but as always we like to start out with unboxings and assemblies we already did the unbo\nxing i highly recommend checking that out if you\'re interested and in this video we\'re going to comp\nlete the assembly so i\'m just going to basically walk through the process and assemble the chair so \nyou kind of have an idea of what you are getting into if you decide to order it so one very cool thi\nng about secret lab chairs is they give you this massive guide it\'s not just a small sheet a crinkly\n thing that\'s hard to read it\'s got a ton of pictures a bunch of words it\'s really easy to follow th\neir step-by-step process and you\'re not going to lose the assembly inst

In [57]:
len(test.split("#####"))

979

In [68]:
mehmeh = gpt_sentiment_analysis(df["content"][0]) 
print(mehmeh)
# df["sentiment_score"] = mehmeh

0.5
0.5


In [53]:
df["content"]

0      are you curious how much assembly is involved ...
1      d we're about to put the chair together hey gu...
2      ve ordered a brand new secret lab titan evo 20...
3      his product but as always we like to start out...
4      xing i highly recommend checking that out if y...
                             ...                        
974         Chair looks amazing! Stoked to lend my music
975    Great chair some good evolutions but doesnt wo...
976    now thinking to get the 2nd one, and give the ...
977                    one of the best ads ive ever seen
978    This is the most comfortable gaming chair that...
Name: content, Length: 979, dtype: object

In [63]:
len(mehmeh)

885

In [37]:
df["sentiment_score"]

0      [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
1      [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
2      [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
3      [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
4      [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
                             ...                        
974    [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
975    [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
976    [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
977    [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
978    [-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...
Name: sentiment_score, Length: 979, dtype: object

In [11]:
df["content"].str.cat()

'are you curious how much assembly is involved in the brand new secret lab titan evo 2022 stick around we\'re about to put the chair together hey guys it\'s ryan with btod.com due to popular demand we have ordered a brand new secret lab titan evo 2022 we do have plans to do reviews and comparisons on this product but as always we like to start out with unboxings and assemblies we already did the unboxing i highly recommend checking that out if you\'re interested and in this video we\'re going to complete the assembly so i\'m just going to basically walk through the process and assemble the chair so you kind of have an idea of what you are getting into if you decide to order it so one very cool thing about secret lab chairs is they give you this massive guide it\'s not just a small sheet a crinkly thing that\'s hard to read it\'s got a ton of pictures a bunch of words it\'s really easy to follow their step-by-step process and you\'re not going to lose the assembly instructions because t

In [70]:
mehmehtest = df.head(20)

In [81]:
mehmehtest["is_feedback"] = mehmehtest["content"].apply(gpt_feedback_classifier) 

C:\Users\Rald999\AppData\Local\Temp\ipykernel_3092\2167498149.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mehmehtest["is_feedback"] = mehmehtest["content"].apply(gpt_feedback_classifier)


In [82]:
mehmehtest

,content,type,source,url,sentiment_score,is_feedback
0,are you curious how much assembly is involved ...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
1,d we're about to put the chair together hey gu...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
2,ve ordered a brand new secret lab titan evo 20...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
3,his product but as always we like to start out...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
4,xing i highly recommend checking that out if y...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
5,lete the assembly so i'm just going to basical...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
6,you kind of have an idea of what you are getti...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
7,ng about secret lab chairs is they give you th...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
8,thing that's hard to read it's got a ton of p...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
9,eir step-by-step process and you're not going ...,transcript,YouTube,https://www.youtube.com/watch?v=8hkCPH4Vtiw,"[-1, 0.5, 1, 0, -1, 1, 1, 0.5, 1, 1, 1, -1, -1...",not related
